In [3]:
import sys

print(sys.version)
print(sys.executable)

3.12.13 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:01) [Clang 20.1.8 ]
/opt/anaconda3/envs/fightchurn/bin/python


In [4]:
from fightchurn import run_churn_listing
from fightchurn.run_churn_listing import run_listing

print("Imports successful")

Imports successful


In [5]:
from pathlib import Path

output_dir = Path.home() / "Desktop" / "fighting-churn-learning" / "output"
output_dir.mkdir(parents=True, exist_ok=True)

print(output_dir)

/Users/toor/Desktop/fighting-churn-learning/output


In [10]:
run_churn_listing.set_churn_environment(
    "churn",
    "fightchurn_user",
    "helloworld",
    str(output_dir),
)

Setting Environment Variables user=fightchurn_user for db=churn, output path =`/Users/toor/Desktop/fighting-churn-learning/output`


In [11]:
import os

os.cpu_count()

10

In [12]:
run_churn_listing.run_standard_simulation(
    n_parallel=5,
    force=True
)

force: true
dev: false
n_parallel: 5
random_seed: null
live_sim: false
init_customers: 10000
growth_rate: 0.1
acausal_churn: 0.0
satisfy_scale: 1.5
satisfy_base: 2.0
min_age: 12
max_age: 82
age_satisfy: 0.5
discount_prob: 0.0
min_discount: 0.05
max_discount: 0.5
discount_satisfy: 1.0
weekday_scale: -0.2
weekend_scale: 0.2
behave_exp_base: 1.6
util_contrib_scale: 2.0
model: socialnet7
start_date: '2020-01-01'
end_date: '2020-06-01'
utility:
  post: 1
  newfriend: 6
  like: 0.5
  adview: -1.5
  dislike: -1
  unfriend: -10
  message: 1
  reply: 1
  mrr: 0
transition:
  churn:
    scale: 0.135
    offset: 0.5
  upsell:
    scale: 0.0
    offset: 0.0
  downsell:
    scale: 0.0
    offset: 0.0
population:
  web: 0.15
  appstore1: 0.3
  appstore2: 0.55
country:
  AU: 0.04
  AR: 0.03
  BR: 0.05
  CA: 0.04
  CH: 0.01
  CN: 0.15
  CO: 0.02
  DE: 0.04
  DK: 0.01
  ES: 0.01
  FR: 0.03
  GB: 0.04
  GR: 0.01
  ID: 0.01
  IN: 0.04
  IT: 0.01
  JP: 0.05
  KR: 0.03
  MX: 0.04
  NL: 0.01
  'NO': 0.01
  

In [13]:
run_listing(2, 2)


Running chapter 2 listing 2 churn_rate on schema socialnet7
SQL:
----------
set search_path = 'socialnet7'; with 
date_range as ( 
	select  '2020-03-01'::date as start_date, '2020-04-01'::date as end_date
), 
start_accounts as  
(
	select distinct account_id        
	from subscription s inner join date_range d on
		s.start_date <= d.start_date   
		and (s.end_date > d.start_date or s.end_date is null)
),
end_accounts as 
(
	select distinct account_id  
	from subscription s inner join date_range d on
		s.start_date <= d.end_date   
		and (s.end_date > d.end_date or s.end_date is null)
), 
churned_accounts as 
(
	Select s.account_id
	from start_accounts s 
	left outer join end_accounts e on 
		s.account_id=e.account_id  
	where e.account_id is null    
),
start_count as (   
	select 	count(start_accounts.*) as n_start from start_accounts
), 
churn_count as ( 
	select 	count(churned_accounts.*) as n_churn from churned_accounts
)
select 
	n_churn::float/n_start::float as churn_rate, 
	1.0

In [14]:
run_churn_listing.run_listing(
    2,
    [1, 2, 3, 4]
)


Running chapter 2 listing 1 net_retention on schema socialnet7
SQL:
----------
set search_path = 'socialnet7'; with 
date_range as (    
	select  '2020-03-01'::date as start_date, '2020-04-01'::date as end_date
), 
start_accounts as    
(
	select  account_id, sum (mrr) as total_mrr    
	from subscription s inner join date_range d on
		s.start_date <= d.start_date    
		and (s.end_date > d.start_date or s.end_date is null)
	group by account_id    
),
end_accounts as    
(
	select account_id, sum(mrr) as total_mrr      
	from subscription s inner join date_range d on
		s.start_date <= d.end_date    
		and (s.end_date > d.end_date or s.end_date is null)
	group by account_id    
), 
retained_accounts as     
(
	select s.account_id, sum(e.total_mrr) as total_mrr     
	from start_accounts s 
	inner join end_accounts e on s.account_id=e.account_id  
	group by s.account_id    
),
start_mrr as (    
	select sum (start_accounts.total_mrr) as start_mrr from start_accounts
), 
retain_mrr as (    

In [18]:
from pathlib import Path
from getpass import getpass
from fightchurn import run_churn_listing

# Output directory
output_dir = (
    Path.home()
    / "Desktop"
    / "fighting-churn-learning"
    / "output"
)

output_dir.mkdir(parents=True, exist_ok=True)

# Ask for PostgreSQL password without displaying it in the notebook
db_password = getpass("PostgreSQL password: ")

# Set database environment for this Jupyter/Python session
run_churn_listing.set_churn_environment(
    db="churn",
    user="fightchurn_user",
    password=db_password,
    output_dir=str(output_dir),
    host="127.0.0.1"
)

# Listing 3.4:
# Create/register the like_per_month metric name
run_churn_listing.run_listing(
    3,
    4,
    version=1,
    schema="socialnet7"
)

# Listing 3.3:
# Calculate and insert the rolling like metric values
run_churn_listing.run_listing(
    3,
    3,
    version=1,
    schema="socialnet7"
)

PostgreSQL password:  ········


Setting Environment Variables user=fightchurn_user for db=churn, output path =`/Users/toor/Desktop/fighting-churn-learning/output`

Running chapter 3 listing 4 metric_name_insert on schema socialnet7
SQL:
----------
set search_path = 'socialnet7'; insert into metric_name values (0,'like_per_month')
ON CONFLICT DO NOTHING;


----------
RESULT:

Running chapter 3 listing 3 count_metric_insert on schema socialnet7
SQL:
----------
set search_path = 'socialnet7'; with date_vals AS (
 	select i::timestamp as metric_date 
from generate_series('2020-02-02', '2020-05-10', '7 day'::interval) i
)
insert into metric (account_id,metric_time,metric_name_id,metric_value)
select account_id, metric_date, 0,  count(*) AS metric_value
from event e inner join date_vals d
on e.event_time < metric_date 
and e.event_time >= metric_date - interval '28 day'
inner join event_type t on t.event_type_id=e.event_type_id
where t.event_type_name='like'
group by account_id, metric_date
ON CONFLICT DO NOTHING;


------

In [19]:
from fightchurn import run_churn_listing

# Register like_per_month
run_churn_listing.run_listing(
    3,
    4,
    version=1,
    schema="socialnet7"
)

# Calculate and insert like_per_month
run_churn_listing.run_listing(
    3,
    3,
    version=1,
    schema="socialnet7"
)


Running chapter 3 listing 4 metric_name_insert on schema socialnet7
SQL:
----------
set search_path = 'socialnet7'; insert into metric_name values (0,'like_per_month')
ON CONFLICT DO NOTHING;


----------
RESULT:

Running chapter 3 listing 3 count_metric_insert on schema socialnet7
SQL:
----------
set search_path = 'socialnet7'; with date_vals AS (
 	select i::timestamp as metric_date 
from generate_series('2020-02-02', '2020-05-10', '7 day'::interval) i
)
insert into metric (account_id,metric_time,metric_name_id,metric_value)
select account_id, metric_date, 0,  count(*) AS metric_value
from event e inner join date_vals d
on e.event_time < metric_date 
and e.event_time >= metric_date - interval '28 day'
inner join event_type t on t.event_type_id=e.event_type_id
where t.event_type_name='like'
group by account_id, metric_date
ON CONFLICT DO NOTHING;


----------
RESULT:
